## Forcing Check: 
Use this notebook to assess forcing in comparison to model debugging/output.

In [1]:
%matplotlib ipympl
import xarray as xr
import matplotlib.pyplot as plt
import pandas as pd
import os

In [2]:
# Configuration
snotel_site = 'Trial Lake'  #Use the Snotel name
forcings = 's00000000'
forcing_version = 'v0.4.4'  # Version of the forcing data
path_env = 'desktop' # 'desktop' or 'alpine'

In [3]:
# Configure environment
if path_env == 'desktop':
    output_path = 'C:/Users/clmbn/NMT_PhD/data/output' #desktop
    snotel_path = 'C:/Users/clmbn/NMT_PhD/data/snotel/' #desktop
    forcing_path = 'C:/Users/clmbn/NMT_PhD/data/forcing/' #desktop
elif path_env == 'alpine':
    output_path = r'/projects/ccox1@xsede.org/lm4px_modelling/output' #Alpine
    snotel_path = r'/projects/ccox1@xsede.org/data/snotel/'
    forcing_path = r'/projects/ccox1@xsede.org/data/forcing/'
else:
    raise ValueError("Invalid path_env. Choose 'desktop' or 'alpine'.")

In [4]:
# Convert name for output paths
site = snotel_site.replace(' ','').lower()
print(f'Output site: {site}')

Output site: triallake


Load forcing:
Forcing is 10 years of NLDAS data that is then looped for the spinup period.

In [5]:
forcing_10yr = xr.open_dataset(os.path.join(forcing_path, site, f'{site}_spinup_s00000000_{forcing_version}.nc'))

In [6]:
forcing_10yr

<xarray.Dataset> Size: 6MB
Dimensions:           (latitude: 1, longitude: 1, time: 87648)
Coordinates:
  * latitude          (latitude) float64 8B 40.68
  * longitude         (longitude) float64 8B -111.0
  * time              (time) datetime64[ns] 701kB 1980-10-01 ... 1990-09-30T2...
Data variables:
    Tair              (time, latitude, longitude) float64 701kB ...
    Qair              (time, latitude, longitude) float64 701kB ...
    Psurf             (time, latitude, longitude) float64 701kB ...
    Rainf             (time, latitude, longitude) float64 701kB ...
    Snowf             (time, latitude, longitude) float64 701kB ...
    SWdown            (time, latitude, longitude) float64 701kB ...
    LWdown            (time, latitude, longitude) float64 701kB ...
    Wind              (time, latitude, longitude) float64 701kB ...
    reference_height  (latitude, longitude) int64 8B ...
    canopy_height     (latitude, longitude) float64 8B ...

In [7]:
# Extract a subset of the forcing data for checking
forcing_subset = forcing_10yr[['SWdown','Tair','Psurf']].isel(latitude=0, longitude=0).to_dataframe()

# Add a numerical index starting at 1
forcing_subset['debug_index'] = range(1, len(forcing_subset) + 1)

In [8]:
# Display data where debug_index is between two values
forcing_subset[(forcing_subset['debug_index'] >= 87596) & (forcing_subset['debug_index'] <= 87600)]

,SWdown,Tair,Psurf,latitude,longitude,debug_index
time,,,,,,
1990-09-28 19:00:00,545.720,282.79,70829.5,40.68,-110.95,87596
1990-09-28 20:00:00,529.536,283.25,70777.1,40.68,-110.95,87597
1990-09-28 21:00:00,501.666,283.71,70724.1,40.68,-110.95,87598
1990-09-28 22:00:00,406.037,283.17,70785.4,40.68,-110.95,87599
1990-09-28 23:00:00,299.280,282.64,70846.0,40.68,-110.95,87600


In [ ]:
SWdown_short = SWdown['1980-12-31':'1981-01-01']

In [ ]:
def get_spinup_forcing(forcing_ds: xr.Dataset, var_name: str) -> pd.Series:
    '''
    Extract a given variable (at a daily timestep) from the forcing dataset and loop it for the 100 year spinup period.
    Return a pandas Series with the looped data.
    '''
    # If 'precip' get sum of 'Rainf' and 'Snowf'
    if var_name == 'precip':
        forcing_ds = forcing_ds.assign(precip=forcing_ds['Rainf'] + forcing_ds['Snowf'])
    
    # Extract the variable data
    var_df = forcing_ds[var_name].isel(latitude=0, longitude=0).to_dataframe()
    data = var_df[var_name] # Extract the variable column
    data.index = data.index.to_datetimeindex()  # This will give a warning but shouldn't be problematic since the spinup forcing is > 1900

    # Resample to daily frequency
    data_daily = data.resample('D').mean()

    # Loop the data for the spinup period
    spinup_years = int(spin_up[:-1])  # Extract the number of years from the spin_up string
    model_start_year = data_daily.index[-1].year - spinup_years
    model_start = data_daily.index[0].replace(year=model_start_year)

    # Create date range accounting for Julian 1900 leap year if necessary
    if model_start < pd.Timestamp('1900-03-01'):
        model_end = data_daily.index[-1] + pd.Timedelta(days=1)  # Add on a day to account for the leap year in 1900
    else:
        model_end = data_daily.index[-1]
    model_index = pd.date_range(start=model_start, end=model_end, freq='D')

    # Loop the data to fill the spinup period, using 1 extra concatenation to ensure full coverage
    looped = pd.concat([data_daily] * int(spinup_years / 10 + 1), ignore_index=True)
    
    # Ensure the looped data matches the model index length
    looped_values = looped[:len(model_index)].values

    # Shift index by 12 hours to match model output
    model_index = model_index + pd.Timedelta(hours=12)

    output = pd.Series(looped_values, index=model_index)
    output.name = var_name

    return output
    


In [ ]:
# Verify the function works for Tair
# test = get_spinup_forcing(forcing_10yr, 'Tair')
# test2 = forcing_10yr['Tair'].isel(latitude=0, longitude=0).to_series()
# test2.index = test2.index.to_datetimeindex()  # This will give a warning but shouldn't be problematic since the spinup forcing is > 1900
# test2 = test2.resample('D').mean()

Load diagnostics

In [ ]:
output_folder = os.path.join(output_path,f'{site}/{site}_spinup_{spin_up}_{forcings}_{spinup_version}/history/')

In [ ]:
# Find the land daily file name in folder and open
land_daily=None
for filename in os.listdir(output_folder):
    if filename.endswith(".land_daily.nc"):
        land_daily = xr.open_dataset(output_folder + filename)
if not land_daily:
    print('Error: no land daily found')

In [ ]:
# Create a dataframe whose index is the name of the variable and the columns are: dimensions, long_name, units
daily_vars_df = pd.DataFrame({
    'dimensions': [land_daily[var].dims for var in land_daily.data_vars],
    'long_name': [land_daily[var].attrs.get('long_name', '') for var in land_daily.data_vars],
    'units': [land_daily[var].attrs.get('units', '') for var in land_daily.data_vars]
}, index=land_daily.data_vars)

In [ ]:
land_daily

#### Calendar conversion:
Create a new calendar index that is gregorian starting at the same time as the output.

In [ ]:
# Get the first time coordinate from the daily dataset
model_start_str = land_daily['time'].to_index()[0].strftime('%Y-%m-%d 12:00')
model_end_str = land_daily['time'].to_index()[-1].strftime('%Y-%m-%d 12:00')
model_start = pd.Timestamp(model_start_str)

# Create a date range for the model index
# Adjust the end date to account for the leap year in 1900
if model_start < pd.Timestamp('1900-03-01'):
    model_end = pd.Timestamp(model_end_str) + pd.Timedelta(days=1)  # Add on a day to account for the leap year in 1900
else:
    model_end = pd.Timestamp(model_end_str) 

output_index_gregorian = pd.date_range(start=model_start, end=model_end, freq='D')

In [ ]:
# Generate some variables for plotting
plot_start_date = model_start.strftime('%Y-%m-%d 12:00')
plot_end_date = model_start.replace(year=model_start.year + 10).strftime('%Y-%m-%d 12:00')

### Analysis: Temperature
In the daily output, it appears that air temperature and shortwave radiation are the only variables that are directly comparible to the forcing data. Even so, t_ref is 2m air temperature, while the forcing is set to 10m. The shortwave radiation is also partitioned by the model into direct and diffuse components, while the forcing is total shortwave radiation.

Even so, I can make a number of comparisons to see if there is consistency.

In [ ]:
model_surface_data = land_daily[['t_ref', 'snow_frac', 'snow_depth', 'Tgrnd']].isel(grid_index=0).to_dataframe()
#model_surface_data.index = output_index_gregorian
model_surface_data = model_surface_data[['t_ref', 'snow_frac', 'snow_depth', 'Tgrnd']]  #Just get the variables of interest

In [ ]:
# Get the forcing Tair
Tair = get_spinup_forcing(forcing_10yr, 'Tair')

In [ ]:
# Plot just the first 20 years of the Tair forcing and t_ref
fig, ax = plt.subplots(figsize=(12, 6))
Tair[plot_start_date:plot_end_date].plot(ax=ax, label='Tair forcing', color='blue')
model_surface_data.loc[plot_start_date:plot_end_date, ['t_ref','Tgrnd']].plot(ax=ax, label='t_ref')
ax.set_title(f'{snotel_site} - 10 year spinup comparison starting 1986')
ax.set_ylabel('Temperature (K)')


### Analysis: Radiation
The model partitions incoming shortwave radiation into direct and diffuse components, as well as, two bands (VIS, NIR). The forcing data provides total shortwave radiation. To compare, I will sum all model radiation components and compare to the forcing data.

In [ ]:
# Extract SW radiation components from land_daily
swdn_dir = land_daily['swdn_dir'].isel(grid_index=0).to_dataframe()
swdn_dir = swdn_dir.reset_index().pivot(index='time', columns='band', values='swdn_dir')

swdn_diff = land_daily['swdn_dif'].isel(grid_index=0).to_dataframe()
swdn_diff = swdn_diff.reset_index().pivot(index='time', columns='band', values='swdn_dif')

# Combine the direct and diffuse components
swdn_combined = pd.merge(swdn_dir, swdn_diff, left_index=True, right_index=True, suffixes=('_dir', '_dif'))

# Test
swdn_combined.head()

In [ ]:
swdn_combined['swdn_total'] = swdn_combined.sum(axis=1)

In [ ]:
swdn_combined.index = output_index_gregorian

In [ ]:
SWdown = get_spinup_forcing(forcing_10yr, 'SWdown')

In [ ]:
# Plot the SW radiation components
fig, ax = plt.subplots(figsize=(12, 6))
swdn_combined.loc[:,'swdn_total'].plot(ax=ax, color='orange', label='Total SW radiation (model)')
#SWdown[plot_start_date:plot_end_date].plot(ax=ax, color='red', label='SWdown forcing')
ax.set_title(f'{snotel_site} - 10 year spinup SW radiation comparison starting 1986')
ax.set_ylabel('SW radiation (W m$^{-2}$)')
# Add legend
ax.legend()

### Analysis: Snow

In [ ]:
# Plot snow and snow fraction in subplots
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
model_surface_data.loc[plot_start_date:plot_end_date,'snow_depth'].plot(ax=ax1, label='Snow Depth', color='blue')
model_surface_data.loc[plot_start_date:plot_end_date,'snow_frac'].plot(ax=ax2, label='Snow Fraction', color='black')